# Lower-level pipeline

This notebook builds a TMAP one step at a time: SMILES to Morgan fingerprints, fingerprints to MinHash signatures, signatures into an LSHForest, the forest to a kNN graph and an OGDF tree layout, and the layout into an HTML page with `TmapViz`.

This path is useful when you want direct control over:

- MinHash settings
- LSHForest construction
- kNN graph inspection
- layout settings, and layout from the forest or from a precomputed graph

For the shortest workflow, use `TMAP(...).fit(X)` instead.


## 1. Load Molecules And Build Fingerprints

We use the molecules from the main chemistry tutorial (`02_cheminformatics.ipynb`), but the kNN graph is built differently. For a dense 0/1 fingerprint matrix like this one, `TMAP(metric="jaccard").fit(fps)` searches the fingerprints directly with USearch (see `11_usearch_jaccard.ipynb`). The MinHash + LSHForest steps below are what the estimator runs for sets, strings and sparse input. We apply them to fingerprints here so each step is easy to follow.

`return_valid=True` also returns one flag per SMILES. A SMILES that RDKit cannot read gets no fingerprint, so we drop it from the list to keep the SMILES lined up with the rows of `fps`. We read the first 2,000 rows to keep the notebook fast; remove `nrows` to map the whole file.


In [ ]:
import pandas as pd

from tmap.utils import fingerprints_from_smiles, molecular_properties

df = pd.read_csv("../examples/cluster_65053.csv", nrows=2000)
smiles = df["smiles"].tolist()

fps, valid = fingerprints_from_smiles(
    smiles, fp_type="morgan", radius=2, n_bits=2048, return_valid=True
)
smiles = [smi for smi, ok in zip(smiles, valid) if ok]
props = molecular_properties(smiles, properties=["mw", "logp", "n_rings"])

print(f"Loaded {len(smiles):,} molecules")
print(f"Fingerprint matrix: {fps.shape}")


## 2. MinHash

`batch_from_binary_array()` is the usual entry point for dense binary fingerprints.
See `07_minhash_deep_dive.ipynb` for the full set of `from_*` and `batch_from_*` methods.


In [2]:
from tmap import MinHash

mh = MinHash(num_perm=128, seed=42)
signatures = mh.batch_from_binary_array(fps)
single_signature = mh.from_binary_array(fps[0])

print(signatures.shape)
print(single_signature.shape)


(2000, 128)
(128,)


## 3. Build LSHForest

This is the main lower-level index for the Jaccard workflow. `d` is the number of MinHash permutations and `l` the number of prefix trees; each tree indexes `d // l` of the hash values.

We use `l=32`, the value the estimator picks for `d=128` (it uses `l = max(8, d // 4)` up to 1M points). The class default is 8, and every extra tree adds index memory and query work.

`get_knn_graph(k, kc)` returns the `k` nearest neighbors of every point. For each point it collects up to `k * kc` candidates from the trees and keeps the `k` closest by the Jaccard distance estimated from the signatures.


In [ ]:
from tmap import LSHForest

lsh = LSHForest(d=128, l=32)
lsh.batch_add(signatures)
lsh.index()
print(f"LSHForest: {lsh.size} signatures in {lsh.l} trees")

knn = lsh.get_knn_graph(k=20, kc=50)
print(knn.indices.shape)
print(knn.distances.shape)


## 4. Compute The Layout Directly From The Forest

`layout_from_lsh_forest()` is the closest match to the classic TMAP pipeline. It builds the kNN graph with `cfg.k` and `cfg.kc`, computes the minimum spanning tree, and lays the tree out with OGDF. It returns the coordinates `x, y` and the tree edges `s, t`.

The estimator sets `deterministic` and `seed` on its layout for you. The low-level functions leave them unset, so set them yourself to get the same layout on every run.

`fme_iterations` sets the number of force-directed iterations, and `sl_extra_scaling_steps` and `sl_scaling_type` control the scaling rounds at each level of the layout. Older TMAP scripts also set `node_size` and `mmm_repeats`. Those two only apply with `cfg.adaptive = False`, which switches to the original single-shot layout. The default adaptive layout sets the node size per level, and `ns_cap` is its main knob for how spread out the map is.

The estimator also joins a kNN graph that has split into pieces (`connect_components`, see `04_working_with_a_fitted_model.ipynb`). The low-level functions skip that step. If the graph splits, OGDF lays out each piece on its own, and `s, t` has fewer than `len(x) - 1` edges.


In [ ]:
from tmap.layout import LayoutConfig, ScalingType, layout_from_lsh_forest

cfg = LayoutConfig()
cfg.k = 20
cfg.kc = 50
cfg.fme_iterations = 1000  # default
cfg.sl_extra_scaling_steps = 10  # default 2
cfg.sl_scaling_type = ScalingType.RelativeToDrawing  # default
cfg.deterministic = True
cfg.seed = 42

x, y, s, t = layout_from_lsh_forest(lsh, cfg)
print(f"{len(x)} nodes, {len(s)} tree edges")


## 5. Alternative: Layout From A Precomputed kNN Graph

Use this when you want to inspect or modify the graph first.


In [5]:
from tmap.layout import layout_from_knn_graph

x2, y2, s2, t2 = layout_from_knn_graph(knn, cfg)
print(len(x2), len(s2))


2000 1999


## 6. Build A Visualization Manually

Here we create a `TmapViz` object without using the estimator, add the SMILES and three color layouts, and write an HTML file you can open in a browser.


In [ ]:
from tmap.visualization import TmapViz

viz = TmapViz()
viz.title = "Lower-level pipeline"
viz.set_points(x, y)
viz.set_edges(s, t)
viz.add_smiles(smiles)
viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
viz.add_color_layout("Ring Count", props["n_rings"].tolist(), categorical=True, color="tab10")

path = viz.write_html("lower_level_pipeline.html")
print(path)


## 7. When To Use This API

Use the lower-level pipeline when:

- you want direct MinHash control
- you want to inspect the LSH forest or the kNN graph
- you want to lay out an externally built graph
- you want MinHash + LSHForest neighbors for dense fingerprints, which the estimator sends to USearch

Use the estimator when you just want to pass data and get a map.
